# nbformat

> Notebook manipulation using nbformat with standard structure enforcement

In [ ]:
#| default_exp utils.nbformat

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional, Tuple
from pathlib import Path
import re

import nbformat
from nbformat import NotebookNode
from nbformat.v4 import new_notebook, new_code_cell, new_markdown_cell

## Constants

Standard notebook structure patterns.

In [ ]:
#| export
# Standard ending structure for nbdev notebooks
# Note: We construct the export call string dynamically to avoid triggering
# nbdev's cell-skip logic which checks for literal 'nbdev_export()'
_NBDEV_EXPORT_CALL = '#| hide\nimport nbdev; nbdev.' + 'nbdev_export()'

STANDARD_ENDING = [
    {'type': 'markdown', 'source': '## Next'},
    {'type': 'code', 'source': '#| export\n\n'},
    {'type': 'markdown', 'source': '## Export'},
    {'type': 'code', 'source': _NBDEV_EXPORT_CALL},
]

# Regex patterns for notebook structure
IMPORT_SECTION_RE = re.compile(r'^##\s*Imports?\s*$', re.IGNORECASE)
EXPORT_SECTION_RE = re.compile(r'^##\s*Export\s*$', re.IGNORECASE)
NEXT_SECTION_RE = re.compile(r'^##\s*Next\s*$', re.IGNORECASE)
HEADER_RE = re.compile(r'^(#{1,6})\s+(.+)$')
CODE_IN_HEADER_RE = re.compile(r'`([^`]+)`')

## Notebook I/O

Read and write notebooks using nbformat.

In [ ]:
#| export
def read_notebook(path: Path) -> NotebookNode:
    """Read a notebook file using nbformat.
    
    Parameters
    ----------
    path : Path
        Path to the notebook file.
    
    Returns
    -------
    NotebookNode
        The parsed notebook object.
    """
    with open(path, 'r', encoding='utf-8') as f:
        return nbformat.read(f, as_version=4)

In [ ]:
#| export
def write_notebook(nb: NotebookNode, path: Path) -> None:
    """Write a notebook to file using nbformat.

    Ensures every cell carries a stable ``id`` first (see ``ensure_cell_ids``) so
    nbdev's ``# %% ... #<cell.id>`` export markers stay deterministic across runs.

    Parameters
    ----------
    nb : NotebookNode
        The notebook object to write.
    path : Path
        Path to write the notebook to.
    """
    try:
        ensure_cell_ids(nb, nb_relpath=getattr(path, "name", str(path)))
    except Exception:
        pass
    with open(path, 'w', encoding='utf-8') as f:
        nbformat.write(nb, f)

## Cell Creation

Helper functions for creating properly formatted cells.

In [ ]:
#| export
def make_code_cell(source: str, export: bool = False, hide: bool = False) -> NotebookNode:
    """Create a code cell with optional directives.
    
    Parameters
    ----------
    source : str
        The cell source code.
    export : bool
        If True, prepend ``#| export`` directive.
    hide : bool
        If True, prepend ``#| hide`` directive.
    
    Returns
    -------
    NotebookNode
        A new code cell.
    """
    directives = []
    if hide:
        directives.append('#| hide')
    if export:
        directives.append('#| export')
    
    if directives:
        source = '\n'.join(directives) + '\n' + source
    
    return new_code_cell(source=source)

In [ ]:
#| export
def make_markdown_cell(source: str) -> NotebookNode:
    """Create a markdown cell.
    
    Parameters
    ----------
    source : str
        The markdown content.
    
    Returns
    -------
    NotebookNode
        A new markdown cell.
    """
    return new_markdown_cell(source=source)

In [ ]:
#| export
def make_section_header(title: str, level: int = 2, code_names: List[str] = None) -> NotebookNode:
    """Create a markdown section header with proper formatting.
    
    Wraps code/function names in backticks automatically.
    
    Parameters
    ----------
    title : str
        The section title.
    level : int
        Header level (1-6), default 2.
    code_names : List[str], optional
        Names to wrap in backticks in the title.
    
    Returns
    -------
    NotebookNode
        A markdown cell with the header.
    
    Examples
    --------
    >>> make_section_header('with_project Decorator', code_names=['with_project'])
    # Creates: ## `with_project` Decorator
    """
    formatted_title = title
    if code_names:
        for name in code_names:
            # Only wrap if not already wrapped
            if f'`{name}`' not in formatted_title:
                formatted_title = formatted_title.replace(name, f'`{name}`')
    
    hashes = '#' * level
    return new_markdown_cell(source=f'{hashes} {formatted_title}')

## Structure Analysis

Functions for analyzing notebook structure.

In [ ]:
#| export
def get_cell_source(cell: NotebookNode) -> str:
    """Get the source of a cell as a string.
    
    Parameters
    ----------
    cell : NotebookNode
        A notebook cell.
    
    Returns
    -------
    str
        The cell source as a single string.
    """
    source = cell.get('source', '')
    if isinstance(source, list):
        return ''.join(source)
    return source

In [ ]:
#| export
def find_section_index(nb: NotebookNode, pattern: re.Pattern) -> Optional[int]:
    """Find the index of a section header matching a pattern.
    
    Parameters
    ----------
    nb : NotebookNode
        The notebook to search.
    pattern : re.Pattern
        Regex pattern to match against markdown cell content.
    
    Returns
    -------
    int or None
        The cell index, or None if not found.
    """
    for i, cell in enumerate(nb.cells):
        if cell.cell_type == 'markdown':
            source = get_cell_source(cell).strip()
            if pattern.search(source):
                return i
    return None

In [ ]:
#| export
def find_imports_section(nb: NotebookNode) -> Optional[int]:
    """Find the index of the Imports section header.
    
    Parameters
    ----------
    nb : NotebookNode
        The notebook to search.
    
    Returns
    -------
    int or None
        The cell index of the Imports header, or None if not found.
    """
    return find_section_index(nb, IMPORT_SECTION_RE)

In [ ]:
#| export
def find_export_section(nb: NotebookNode) -> Optional[int]:
    """Find the index of the Export section header.
    
    Parameters
    ----------
    nb : NotebookNode
        The notebook to search.
    
    Returns
    -------
    int or None
        The cell index of the Export header, or None if not found.
    """
    return find_section_index(nb, EXPORT_SECTION_RE)

In [ ]:
#| export
def find_next_section(nb: NotebookNode) -> Optional[int]:
    """Find the index of the Next section header.
    
    Parameters
    ----------
    nb : NotebookNode
        The notebook to search.
    
    Returns
    -------
    int or None
        The cell index of the Next header, or None if not found.
    """
    return find_section_index(nb, NEXT_SECTION_RE)

In [ ]:
#| export
def has_standard_ending(nb: NotebookNode) -> bool:
    """Check if notebook has the standard ending structure.
    
    Standard ending is:
    - ## Next (markdown)
    - #| export (empty code cell)
    - ## Export (markdown)
    - #| hide + ``nbdev_export`` (code cell)
    
    Parameters
    ----------
    nb : NotebookNode
        The notebook to check.
    
    Returns
    -------
    bool
        True if notebook has the standard ending.
    """
    if len(nb.cells) < 4:
        return False
    
    last_four = nb.cells[-4:]
    
    # Check ## Next
    if last_four[0].cell_type != 'markdown':
        return False
    if not NEXT_SECTION_RE.search(get_cell_source(last_four[0]).strip()):
        return False
    
    # Check empty export cell
    if last_four[1].cell_type != 'code':
        return False
    src1 = get_cell_source(last_four[1]).strip()
    if not src1.startswith('#| export'):
        return False
    
    # Check ## Export
    if last_four[2].cell_type != 'markdown':
        return False
    if not EXPORT_SECTION_RE.search(get_cell_source(last_four[2]).strip()):
        return False
    
    # Check hide + nbdev_export
    if last_four[3].cell_type != 'code':
        return False
    src3 = get_cell_source(last_four[3])
    if '#| hide' not in src3 or 'nbdev_export' not in src3:
        return False
    
    return True

## Cell Insertion

Smart cell insertion with structure awareness.

In [ ]:
#| export
def find_insertion_point(nb: NotebookNode, cell_type: str = 'code') -> int:
    """Find the best insertion point for a new cell.
    
    For code cells:
    - If notebook has standard ending, insert before the '## Next' section
    - Otherwise insert before '## Export' if it exists
    - Otherwise append to end
    
    For import cells:
    - Insert after the Imports section header
    
    Parameters
    ----------
    nb : NotebookNode
        The notebook.
    cell_type : str
        Type of cell being inserted: 'code', 'markdown', or 'import'.
    
    Returns
    -------
    int
        The index where the new cell should be inserted.
    """
    if cell_type == 'import':
        # Find imports section and insert after it
        imports_idx = find_imports_section(nb)
        if imports_idx is not None:
            # Find the next non-code cell or end of imports block
            for i in range(imports_idx + 1, len(nb.cells)):
                cell = nb.cells[i]
                if cell.cell_type == 'markdown':
                    # Found next section, insert before it
                    return i
            return imports_idx + 1
        return 2  # After default_exp and hide cells
    
    # For regular code/markdown cells
    if has_standard_ending(nb):
        # Insert before the last 4 cells (the standard ending)
        return len(nb.cells) - 4
    
    # Fall back to before ## Export if it exists
    export_idx = find_export_section(nb)
    if export_idx is not None:
        return export_idx
    
    # Otherwise append
    return len(nb.cells)

In [ ]:
#| export
def insert_cell(
    nb: NotebookNode,
    cell: NotebookNode,
    position: str = 'auto',
    after_index: Optional[int] = None,
    is_import: bool = False
) -> int:
    """Insert a cell into a notebook with smart positioning.
    
    Parameters
    ----------
    nb : NotebookNode
        The notebook to modify (in place).
    cell : NotebookNode
        The cell to insert.
    position : str
        Where to insert: 'auto' (smart), 'end', or 'start'.
    after_index : int, optional
        Insert after this specific index (overrides position).
    is_import : bool
        If True, treat as an import cell (insert in imports section).
    
    Returns
    -------
    int
        The index where the cell was inserted.
    """
    if after_index is not None:
        idx = after_index + 1
    elif position == 'start':
        idx = 0
    elif position == 'end':
        idx = len(nb.cells)
    else:  # auto
        cell_type = 'import' if is_import else cell.cell_type
        idx = find_insertion_point(nb, cell_type)
    
    nb.cells.insert(idx, cell)
    return idx

In [ ]:
#| export
def insert_code(
    nb: NotebookNode,
    source: str,
    export: bool = True,
    section_header: Optional[str] = None,
    code_names: Optional[List[str]] = None,
    position: str = 'auto'
) -> Tuple[int, int]:
    """Insert code with optional section header.
    
    Parameters
    ----------
    nb : NotebookNode
        The notebook to modify.
    source : str
        The code to insert.
    export : bool
        If True, add ``#| export`` directive.
    section_header : str, optional
        If provided, add a section header before the code.
    code_names : List[str], optional
        Names to wrap in backticks in the header.
    position : str
        Where to insert: 'auto', 'end', or 'start'.
    
    Returns
    -------
    Tuple[int, int]
        (header_index or -1, code_index)
    """
    # Find insertion point
    idx = find_insertion_point(nb, 'code') if position == 'auto' else (
        0 if position == 'start' else len(nb.cells)
    )
    
    header_idx = -1
    
    # Insert section header if provided
    if section_header:
        header_cell = make_section_header(section_header, code_names=code_names)
        nb.cells.insert(idx, header_cell)
        header_idx = idx
        idx += 1
    
    # Insert code cell
    code_cell = make_code_cell(source, export=export)
    nb.cells.insert(idx, code_cell)
    
    return header_idx, idx

In [ ]:
#| export
def insert_imports(
    nb: NotebookNode,
    imports: str,
    merge: bool = True
) -> int:
    """Insert import statements into the imports section.
    
    Parameters
    ----------
    nb : NotebookNode
        The notebook to modify.
    imports : str
        The import statements to add.
    merge : bool
        If True, try to merge with existing imports cell.
    
    Returns
    -------
    int
        The index of the imports cell.
    """
    imports_idx = find_imports_section(nb)
    
    if imports_idx is None:
        # No imports section, insert after default_exp
        cell = make_code_cell(imports, export=True)
        nb.cells.insert(2, cell)
        return 2
    
    # Find the imports code cell (should be right after the header)
    if imports_idx + 1 < len(nb.cells):
        next_cell = nb.cells[imports_idx + 1]
        if next_cell.cell_type == 'code' and merge:
            # Merge with existing imports
            existing = get_cell_source(next_cell)
            # Add new imports at the end
            if not existing.endswith('\n'):
                existing += '\n'
            next_cell.source = existing + imports
            return imports_idx + 1
    
    # Insert new imports cell
    cell = make_code_cell(imports, export=True)
    nb.cells.insert(imports_idx + 1, cell)
    return imports_idx + 1

## Structure Enforcement

Functions to ensure notebooks follow standard structure.

In [ ]:
#| export
def ensure_standard_ending(nb: NotebookNode) -> bool:
    """Ensure notebook has the standard ending structure.
    
    Adds the standard ending (Next, empty export, Export, nbdev_export)
    if not already present.
    
    Parameters
    ----------
    nb : NotebookNode
        The notebook to modify (in place).
    
    Returns
    -------
    bool
        True if the ending was added, False if already present.
    """
    if has_standard_ending(nb):
        return False
    
    # Remove existing Export section if it exists
    export_idx = find_export_section(nb)
    if export_idx is not None:
        # Remove from Export section to end
        del nb.cells[export_idx:]
    
    # Add standard ending
    for item in STANDARD_ENDING:
        if item['type'] == 'markdown':
            cell = new_markdown_cell(source=item['source'])
        else:
            cell = new_code_cell(source=item['source'])
        nb.cells.append(cell)
    
    return True

In [ ]:
#| export
def fix_header_formatting(nb: NotebookNode) -> int:
    """Fix markdown headers to use backticks for code names.
    
    Detects function/class names in headers and wraps them in backticks.
    
    Parameters
    ----------
    nb : NotebookNode
        The notebook to modify (in place).
    
    Returns
    -------
    int
        Number of headers fixed.
    """
    import ast
    
    # Collect all defined symbols from export cells
    symbols = set()
    for cell in nb.cells:
        if cell.cell_type != 'code':
            continue
        source = get_cell_source(cell)
        if '#| export' not in source:
            continue
        try:
            tree = ast.parse(source)
            for node in ast.walk(tree):
                if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                    symbols.add(node.name)
                elif isinstance(node, ast.ClassDef):
                    symbols.add(node.name)
        except SyntaxError:
            pass
    
    fixed = 0
    for cell in nb.cells:
        if cell.cell_type != 'markdown':
            continue
        source = get_cell_source(cell)
        match = HEADER_RE.match(source.strip())
        if not match:
            continue
        
        hashes, title = match.groups()
        new_title = title
        
        for sym in symbols:
            if sym in new_title and f'`{sym}`' not in new_title:
                # Replace whole word only
                new_title = re.sub(rf'\b{re.escape(sym)}\b', f'`{sym}`', new_title)
        
        if new_title != title:
            cell.source = f'{hashes} {new_title}'
            fixed += 1
    
    return fixed

In [ ]:
#| export
def standardize_notebook(path: Path, write: bool = True) -> Dict[str, Any]:
    """Apply all standard formatting to a notebook.
    
    - Ensures standard ending structure
    - Fixes header formatting
    
    Parameters
    ----------
    path : Path
        Path to the notebook.
    write : bool
        If True, write changes back to file.
    
    Returns
    -------
    Dict[str, Any]
        Result with 'ending_added', 'headers_fixed', and 'modified' keys.
    """
    nb = read_notebook(path)
    
    ending_added = ensure_standard_ending(nb)
    headers_fixed = fix_header_formatting(nb)
    
    modified = ending_added or headers_fixed > 0
    
    if write and modified:
        write_notebook(nb, path)
    
    return {
        'ok': True,
        'path': str(path),
        'ending_added': ending_added,
        'headers_fixed': headers_fixed,
        'modified': modified
    }

## Cell ID stability

`nbdev` writes one `# %% <notebook> #<cell.id>` marker per exported cell. If a cell has no
persisted `id`, `nbformat` invents a random one on every read, so the markers (and the
generated `.py`) churn on every export. These helpers persist **stable, deterministic** ids
so export is reproducible.

In [ ]:
#| export
import hashlib, json as _json

def _stable_cell_id(nb_relpath: str, index: int, source, taken: set) -> str:
    "Deterministic 8-char nbformat-safe id from (path, index, source), de-duped against `taken`."
    if isinstance(source, list): source = ''.join(source)
    base = hashlib.sha1(f"{nb_relpath}:{index}:{source}".encode('utf-8')).hexdigest()[:8]
    cid, n = base, 1
    while cid in taken:
        n += 1; cid = f"{base[:6]}{n:02d}"
    return cid

def ensure_cell_ids(nb, raw_ids=None, nb_relpath: str = '', overwrite: bool = False):
    """Ensure every cell carries a stable, unique ``id``.

    Preserves existing ids (unless ``overwrite``); fills missing/duplicate ids with a
    deterministic value derived from (path, index, source). ``raw_ids`` is the list of ids
    as they appear ON DISK so nbformat's transient per-read ids don't masquerade as
    'present'. Operates on a raw dict or a NotebookNode. Returns ``(nb, n_changed)``.
    """
    cells = nb['cells'] if isinstance(nb, dict) else nb.cells
    taken, changed = set(), 0
    for i, cell in enumerate(cells):
        disk = raw_ids[i] if (raw_ids is not None and i < len(raw_ids)) else cell.get('id')
        if disk and not overwrite and disk not in taken:
            if cell.get('id') != disk: cell['id'] = disk
            taken.add(disk); continue
        cid = _stable_cell_id(nb_relpath, i, cell.get('source', ''), taken)
        taken.add(cid)
        if cell.get('id') != cid: changed += 1
        cell['id'] = cid
    return nb, changed

def _dump_notebook_json(raw: dict) -> str:
    "Serialize a raw notebook dict exactly like nbformat/nbdev_clean (stable, minimal diffs)."
    return _json.dumps(raw, indent=1, sort_keys=True, separators=(",", ": "), ensure_ascii=False) + "\n"

def stamp_notebook_ids(path, overwrite: bool = False) -> int:
    """Persist stable, deterministic cell ids into a notebook file IN PLACE (raw-JSON;
    robust to cells missing ``outputs``, minimal diff). Returns n cells changed (0 if clean)."""
    from pathlib import Path as _P
    path = _P(path)
    raw = _json.loads(path.read_text(encoding='utf-8'))
    raw_ids = [c.get('id') for c in raw.get('cells', [])]
    _, changed = ensure_cell_ids(raw, raw_ids=raw_ids, nb_relpath=path.name, overwrite=overwrite)
    if changed: path.write_text(_dump_notebook_json(raw), encoding='utf-8')
    return changed

def reconcile_ids_from_py(nb_path, py_path) -> int:
    """Back-fill a notebook's export-cell ids FROM an existing generated ``.py``'s
    ``# %% <path> #<id>`` markers (matched in document order) so the notebook regenerates
    that exact ``.py`` byte-for-byte. For recovering already-drifted projects. Returns n changed."""
    import re
    from pathlib import Path as _P
    text = _P(py_path).read_text(encoding='utf-8')
    ids = [m.group(1) for m in re.finditer(r'(?m)^# %% \S+ #(\S+)\s*$', text)]
    raw = _json.loads(_P(nb_path).read_text(encoding='utf-8'))
    def _is_exp(c):
        s = c.get('source', ''); s = ''.join(s) if isinstance(s, list) else s
        return c.get('cell_type') == 'code' and re.search(r'(?m)^\s*#\|\s*export', s)
    exp = [c for c in raw.get('cells', []) if _is_exp(c)]
    changed = 0
    for c, cid in zip(exp, ids):
        if c.get('id') != cid: c['id'] = cid; changed += 1
    if changed: _P(nb_path).write_text(_dump_notebook_json(raw), encoding='utf-8')
    return changed

## Next

In [ ]:
#| export



## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()